In [ ]:
# Configuration
#benchmark_csv_file = "S:\\MasterThesisData\\BenchmarkResults\\SLIC\\BreastUltrasoundMalignant_Compactness-Count-Iterations-InitialRadius.csv"
benchmark_csv_file = "S:\\MasterThesisData\\BenchmarkResults\\SLIC\\BrainMRI_Compactness-Count-Iterations-InitialRadius.csv"
delimiter = ";"

special_parameter_1 = "Superpixel Count"
special_parameter_2 = "Compactness"

quality_metric = "Dice Coefficient"

ignore_zero_dice_coefficient = True

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

hist_cols = [f"OverlapPercentage_{i}" for i in range(101)]

# Read CSV with header
df = pd.read_csv(benchmark_csv_file, sep=delimiter)

# Select relevant columns by name
parsed = df[[special_parameter_1, special_parameter_2, quality_metric, *hist_cols]].copy()
parsed = parsed.apply(pd.to_numeric, errors="coerce")

# Filter out rows where quality = 0
if ignore_zero_dice_coefficient:
    parsed = parsed[parsed[quality_metric] > 0]

print(f"Total valid rows: {len(parsed)}")

# Average out the quality metric for each combination of parameters
parsed = parsed.groupby([special_parameter_1, special_parameter_2], as_index=False)[quality_metric].mean()

# Build grid for surface plot
grid = parsed.pivot(index=special_parameter_2, columns=special_parameter_1, values=quality_metric)
x = grid.columns.to_numpy()
y = grid.index.to_numpy()
X, Y = np.meshgrid(x, y)
Z = grid.to_numpy()

# 2.5D surface plot
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(X, Y, Z, cmap="viridis", edgecolor="none")

ax.set_xlabel(special_parameter_1)
ax.set_ylabel(special_parameter_2)
ax.set_zlabel(quality_metric)
ax.view_init(elev=30, azim=225)
fig.colorbar(surf, ax=ax, shrink=0.6, pad=0.1, label=quality_metric)

plt.tight_layout()
plt.show()


In [ ]:
# Average quality metric over param2 for each param1 value
mean_by_param1 = grid.mean(axis=0)

# Average quality metric over param1 for each param2 value
mean_by_param2 = grid.mean(axis=1)

fig_avg, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(mean_by_param1.index, mean_by_param1.values, marker="o", linewidth=1.8)
axes[0].set_title(f"Avg {quality_metric} vs {special_parameter_1}")
axes[0].set_xlabel(special_parameter_1)
axes[0].set_ylabel(f"Avg {quality_metric}")
axes[0].grid(alpha=0.3)

axes[1].plot(mean_by_param2.index, mean_by_param2.values, marker="o", linewidth=1.8)
axes[1].set_title(f"Avg {quality_metric} vs {special_parameter_2}")
axes[1].set_xlabel(special_parameter_2)
axes[1].set_ylabel(f"Avg {quality_metric}")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Parallel coordinates view of the same quality metric surface data
pc_df = grid.reset_index().copy()
pc_df[special_parameter_2] = pc_df[special_parameter_2].astype(int).astype(str)

ax_pc = pd.plotting.parallel_coordinates(
	pc_df,
	class_column=special_parameter_2,
	colormap="viridis",
	linewidth=1.8,
	alpha=0.9
)

ax_pc.set_title(f"{quality_metric} for multiple {special_parameter_2} and {special_parameter_1}")
ax_pc.set_xlabel(special_parameter_1)
ax_pc.set_ylabel(quality_metric)
plt.xticks(rotation=45)
plt.legend(title=special_parameter_2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# Parameter 2 threshold
threshold = 20

# Re-read full benchmark data to recover histogram bins
hist_cols = [f"OverlapPercentage_{i}" for i in range(1, 99, 1)]
full = pd.read_csv(benchmark_csv_file, sep=delimiter)
full = full[["Actual Superpixel Count", special_parameter_1, special_parameter_2, quality_metric, *hist_cols]].copy()
full = full.apply(pd.to_numeric, errors="coerce")

# Filter
if ignore_zero_dice_coefficient:
	full = full[full[quality_metric] > 0]

def build_hist_df(subset, label):
	hist_by_param1 = subset.groupby(special_parameter_1)[hist_cols + ["Actual Superpixel Count"]].mean()
	avg_sp_count = hist_by_param1["Actual Superpixel Count"]
	hist_normalized = hist_by_param1[hist_cols].div(avg_sp_count, axis=0)
	pc_df = hist_normalized.reset_index().copy()
	pc_df[special_parameter_1] = pc_df[special_parameter_1].astype(int).astype(str)
	rename_map = {col: int(col.split("_")[1]) for col in hist_cols}
	pc_df = pc_df.rename(columns=rename_map)
	return pc_df

below = full[full[special_parameter_2] < threshold]
above = full[full[special_parameter_2] >= threshold]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

for ax, subset, label in zip(axes, [below, above], [f"{special_parameter_2} < {threshold}", f"{special_parameter_2} >= {threshold}"]):
	pc_df = build_hist_df(subset, label)
	pd.plotting.parallel_coordinates(
		pc_df,
		class_column=special_parameter_1,
		colormap="viridis",
		linewidth=1.5,
		alpha=0.85,
		ax=ax
	)
	ax.set_title(f"Distribution of SP overlap with GT mask per {special_parameter_1} ({label})")
	ax.set_xlabel("Percentage of SP mask in GT mask")
	ax.set_ylabel("Relative SP count")
	ax.legend(title=special_parameter_1, bbox_to_anchor=(1.02, 1), loc="upper left")
	plt.setp(ax.get_xticklabels(), rotation=90)

plt.tight_layout()
plt.show()
